# 14: Subgroups and group-conditional certificates (E5, RQ5)

Design decisions
- One fixed model per task, the frozen one: CDC Diabetes and BRFSS Heart disease (notebooks 05, 08, 09) and Cardio-70k (notebooks 09c, 09d), with their candidate sets, box tables, explained subsamples and targets. Every hash is checked before use.
- Groups: sex; age bands 18 to 44, 45 to 64, 65 or older; income tertiles for the two BRFSS tasks, cut on the training portion at the codes whose groups come closest to thirds. Race and ethnicity are not in any of the three public tables. No Cardio patient is 65 or older, so that group is left out.
- Methods: marginal SCAX (the frozen release); group-conditional SCAX (Eq. 11), standard version per attribute with rho(g) = 1/|G|, and strict version in which every group of the patient must pass, with rho(g) = 1/(all groups); SGR without explanations, one threshold per kind and, as a group-wise comparison, one per kind and group. The release order is unchanged, with the group bound in place of the marginal bound. A box that a group bound would release is re-checked by the second exact check first.
- Group policy tests on the explained patients of calibration split 2 at delta_2/(2|G|) per kind and group. Results on the explained test patients at the frozen targets: certified share per group, selective risk with one-sided 95% Clopper-Pearson bounds, worst-group risk, the gap between the highest and lowest certified share (Newcombe 95% interval) and the coverage cost against marginal SCAX.
- Group counts on calibration split 1 and second-check verdicts are saved on Drive, so an interrupted run resumes.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public, splits
from scax.experiments.main_results import FrozenTask, pair_arrays
from scax.experiments import subgroups as sg
from scax.explain.candidates import CandidateGenerator
from scax.explain.verify import verify_boxes
from scax.baselines.selective import sgr

SEED = 42
seed_everything(SEED)
LIMIT = None          # every frozen explained patient
TASKS = ['cdc_diabetes', 'brfss2015_heart', 'cardio70k']
TASK_NAMES = {'cdc_diabetes': 'CDC Diabetes', 'brfss2015_heart': 'BRFSS Heart disease', 'cardio70k': 'Cardio-70k'}
REC05, REC08 = json.load(open(f'{paths.RESULTS}/05_frozen.json')), json.load(open(f'{paths.RESULTS}/08_certificates.json'))
REC09C, REC09D = json.load(open(f'{paths.RESULTS}/09c_frozen.json')), json.load(open(f'{paths.RESULTS}/09d_results.json'))
assert LIMIT is not None or REC09D['limit'] is None, 'notebook 09d must have run on every explained patient'
FROZEN_SHA = {**REC05['frozen_config_sha256'], 'cardio70k': REC09C['frozen_config_sha256']['cardio70k']}
BOX_SHA = {**REC08['box_table_sha256'], 'cardio70k': REC09D['box_table_sha256']}
FT = {t: FrozenTask(t, REPO_DIR, DRIVE_ROOT, FROZEN_SHA[t]) for t in TASKS}
CERT_DIR = paths.DERIVED / 'certificates'
SUB_DIR = paths.DERIVED / 'subgroups'; SUB_DIR.mkdir(parents=True, exist_ok=True)
for t in TASKS:
    T = FT[t].targets
    print(f"{TASK_NAMES[t]}: frozen configuration {FT[t].cfg_sha[:16]}, alpha_0 {T['alpha_0']}, alpha_1 {T['alpha_1']}, "
          f"alpha_sel {T['alpha_sel']}")

## Groups and group sizes

In [ ]:
DATA, GROUPS, SIZE_ROWS, CUTS = {}, {}, [], {}
for task in TASKS:
    ft = FT[task]
    X, y = public.LOADERS[task](paths.RAW)
    names = list(X.columns); Xa = X.to_numpy(dtype=float); del X
    code = np.load(paths.DERIVED / 'splits' / f'{task}.npy')
    assert splits.summary(code)['sha256'] == ft.cfg['split_sha256'], 'split differs from the frozen one'
    train = splits.indices(code, 'train')
    cuts = sg.income_cuts(Xa[train, names.index('Income')]) if 'Income' in names else None
    CUTS[task] = cuts
    allg = sg.groups_for(task, Xa, names, cuts)
    G = sg.drop_empty(allg, {a: (c[train], l) for a, (c, l) in allg.items()})
    for a, (c, l) in allg.items():
        for lab in l:
            if lab not in G[a][1]:
                print(f'{TASK_NAMES[task]}: {lab} has no record in the training portion and is left out')
    if task == 'cardio70k':
        h = Xa[train, names.index('height')]; s = G['sex'][0][train]
        print(f"Cardio-70k mean height: code 1 {h[s == 0].mean():.1f} cm, code 2 {h[s == 1].mean():.1f} cm (dataset card: 1 women, 2 men)")
    cal2 = ft.cal2_rows if LIMIT is None else ft.cal2_rows[:LIMIT]
    test = ft.test_rows if LIMIT is None else ft.test_rows[:LIMIT]
    DATA[task] = {'Xa': Xa, 'y': y, 'code': code, 'cal2': cal2, 'test': test}
    GROUPS[task] = G
    cal1 = splits.indices(code, 'cal1')
    for a, (c, l) in G.items():
        for g, lab in enumerate(l):
            SIZE_ROWS.append({'task': task, 'attribute': sg.ATTRIBUTE_NAMES[a], 'group': lab,
                              'training_portion': int((c[train] == g).sum()), 'calibration_split_1': int((c[cal1] == g).sum()),
                              'outcome_share_calibration_split_1': float(y[cal1][c[cal1] == g].mean()),
                              'explained_calibration_split_2': int((c[cal2] == g).sum()), 'explained_test': int((c[test] == g).sum())})
SIZES = pd.DataFrame(SIZE_ROWS)
SIZES.to_csv(f'{paths.RESULTS}/14_group_sizes.csv', index=False)
print({t: c for t, c in CUTS.items()})
print(SIZES.round(3).to_string(index=False))

## Frozen box tables and candidate sets

In [ ]:
STATE = {}
for task in TASKS:
    ft, D = FT[task], DATA[task]
    f = CERT_DIR / f'{task}__boxes.parquet'
    assert sha256_file(f) == BOX_SHA[task], 'box table differs from the frozen one'
    df = pd.read_parquet(f)
    c = ft.cfg['candidates']
    gen = CandidateGenerator(ft.pred, M=c['M'], seed=c['order_seed'], node_budget=c['node_budget'])
    assert gen.load(CERT_DIR / f'{task}__candidates.pkl')
    keys = [tuple(tuple(t) for t in json.loads(k)) for k in df.key]
    cls = df.box_class.to_numpy().astype(int)
    box_index = {(k, int(cl)): i for i, (k, cl) in enumerate(zip(keys, cls))}
    parts = {}
    for name in ('cal2', 'test'):
        rows = D[name]
        C = ft.grid.cells(D['Xa'][rows]); yhat = ft.pred.predict_cells(C).astype(int)
        pp, pb = pair_arrays([gen.cache[gen.cell_key(cc)] for cc in C], box_index, ft.M)
        parts[name] = {'rows': rows, 'yhat': yhat, 'y': D['y'][rows], 'pp': pp, 'pb': pb,
                       'codes': {a: GROUPS[task][a][0][rows] for a in GROUPS[task]}}
    del gen
    STATE[task] = {'keys': keys, 'cls': cls, 'size': df['size'].to_numpy(), 'n_B': df.n_B.to_numpy(), 'U': df.U.to_numpy(),
                   'log_prior': df.log_prior.to_numpy(), 'sound': df.verified.to_numpy() == 'sound', 'parts': parts}
    print(f"{TASK_NAMES[task]}: {len(keys)} boxes, {int(STATE[task]['sound'].sum())} confirmed by the second check, "
          f"{len(parts['cal2']['rows'])} + {len(parts['test']['rows'])} explained patients")

## Group counts on calibration split 1 and group-conditional bounds (Eq. 11)

In [ ]:
for task in TASKS:
    ft, D, st = FT[task], DATA[task], STATE[task]
    G = GROUPS[task]
    n_all = sum(len(l) for _, l in G.values())
    cache = SUB_DIR / f'{task}__group_counts.npz'
    t0 = time.time()
    if cache.exists() and str(np.load(cache)['box_sha']) == BOX_SHA[task]:
        z = np.load(cache)
        NK = {a: (z[f'n_{a}'], z[f'k_{a}']) for a in G}
        print(f'{TASK_NAMES[task]}: group counts loaded from Drive')
    else:
        cal1 = splits.indices(D['code'], 'cal1')
        C1, y1 = ft.grid.cells(D['Xa'][cal1]), D['y'][cal1]
        NK = {a: sg.group_counts(st['keys'], st['cls'], C1, y1, G[a][0][cal1], len(G[a][1])) for a in G}
        np.savez_compressed(cache, box_sha=BOX_SHA[task], **{f'n_{a}': NK[a][0] for a in G}, **{f'k_{a}': NK[a][1] for a in G})
        del C1
    d1 = ft.targets['delta_1']
    st['U_std'] = {a: sg.group_bounds(*NK[a], st['log_prior'], d1, 1 / len(G[a][1])) for a in G}
    st['U_strict'] = {a: sg.group_bounds(*NK[a], st['log_prior'], d1, 1 / n_all) for a in G}
    st['n_groups_all'] = n_all
    print(f"{TASK_NAMES[task]}: {n_all} groups, bounds for {len(st['keys'])} boxes ({time.time() - t0:.0f}s)")

## Second exact check of boxes that a group bound would release

In [ ]:
VERIFY = {}
for task in TASKS:
    ft, st = FT[task], STATE[task]
    T = ft.targets
    alpha = {0: T['alpha_0'], 1: T['alpha_1']}
    need = sg.needs_check(list(st['U_std'].values()) + list(st['U_strict'].values()), st['cls'], alpha, st['sound'])
    vfile = SUB_DIR / f'{task}__verified.json'
    done = json.load(open(vfile)) if vfile.exists() else {}
    if done.get('box_sha') != BOX_SHA[task]:
        done = {'box_sha': BOX_SHA[task], 'verdicts': {}}
    todo = [int(i) for i in need if str(int(i)) not in done['verdicts']]
    t0 = time.time()
    for s in range(0, len(todo), 2000):
        part = todo[s:s + 2000]
        verdicts, _ = verify_boxes(ft.grid, [(st['keys'][i], int(st['cls'][i])) for i in part])
        done['verdicts'].update({str(i): verdicts[(st['keys'][i], int(st['cls'][i]))] for i in part})
        json.dump(done, open(vfile, 'w'))
    for i in need:
        st['sound'][int(i)] = done['verdicts'][str(int(i))] == 'sound'
    VERIFY[task] = {'boxes_checked': int(len(need)), 'sound': int(sum(done['verdicts'][str(int(i))] == 'sound' for i in need)),
                    'unknown': int(sum(done['verdicts'][str(int(i))] == 'unknown' for i in need)), 'seconds': time.time() - t0}
    print(TASK_NAMES[task], VERIFY[task])

## Releases: marginal and group-conditional SCAX

In [ ]:
METHODS = ['SCAX, marginal', 'SCAX, group-conditional', 'SCAX, group-conditional, strict']
REL = {}
for task in TASKS:
    ft, st = FT[task], STATE[task]
    T = ft.targets
    alpha = {0: T['alpha_0'], 1: T['alpha_1']}
    G = GROUPS[task]
    REL[task] = {}
    for name, p in st['parts'].items():
        pp, pb, n = p['pp'], p['pb'], len(p['rows'])
        ok = st['sound'][pb]
        args = (n, st['size'], st['cls'], st['n_B'], alpha)
        r = {'SCAX, marginal': sg.release_pairs(pp, pb, np.where(ok, st['U'][pb], 1.0), *args)}
        for a in G:
            r[('SCAX, group-conditional', a)] = sg.release_pairs(pp, pb, np.where(ok, sg.pair_bound(st['U_std'][a], p['codes'][a], pp, pb), 1.0), *args)
        strict = np.max([sg.pair_bound(st['U_strict'][a], p['codes'][a], pp, pb) for a in G], axis=0)
        r['SCAX, group-conditional, strict'] = sg.release_pairs(pp, pb, np.where(ok, strict, 1.0), *args)
        REL[task][name] = r
    # the marginal release must reproduce the frozen main results
    m = pd.read_csv(f"{paths.RESULTS}/{'09d' if task == 'cardio70k' else '09'}_main.csv")
    m = m[m.frozen_targets & (m.task == task)]
    pt = st['parts']['test']
    for k in (0, 1):
        acc = int(((REL[task]['test']['SCAX, marginal'] >= 0) & (pt['yhat'] == k)).sum())
        ref = int(m[m.kind == sg.KIND_NAMES[k]].test_accepted.iloc[0])
        assert LIMIT is not None or acc == ref, f'{task}: marginal release differs from the frozen results ({acc} against {ref})'
    print(TASK_NAMES[task], 'releases done; marginal release matches the frozen main results' if LIMIT is None else 'releases done')

## SGR without explanations, one threshold per kind and per kind and group

In [ ]:
SGR_REL, SGR_INFO = {}, []
for task in TASKS:
    ft, D, st = FT[task], DATA[task], STATE[task]
    T = ft.targets
    alpha_sel = {0: T['alpha_sel']['negative_outcome'], 1: T['alpha_sel']['enrichment']}
    lab = np.r_[splits.indices(D['code'], 'cal1'), st['parts']['cal2']['rows']]
    m_l = ft.pred.score(D['Xa'][lab]); yh_l = (m_l > ft.ens.tau).astype(int); y_l = D['y'][lab]
    pt = st['parts']['test']
    m_t = ft.pred.score(D['Xa'][pt['rows']])
    assert np.array_equal((m_t > ft.ens.tau).astype(int), pt['yhat'])
    G = GROUPS[task]
    codes_l = {a: G[a][0][lab] for a in G}

    def fit_release(sel_l, sel_t, delta, k):
        conf = m_l[sel_l] if k == 1 else -m_l[sel_l]
        res = sgr(conf, (y_l[sel_l] != k).astype(int), alpha_sel[k], delta)
        tconf = m_t if k == 1 else -m_t
        return sel_t & (tconf >= res['theta']), res

    rel = {'SGR': np.full(len(pt['rows']), -1, dtype=np.int64)}
    for k in (0, 1):
        hit, res = fit_release(yh_l == k, pt['yhat'] == k, T['delta_2'] / 2, k)
        rel['SGR'][hit] = 0
        SGR_INFO.append({'task': task, 'method': 'SGR', 'attribute': None, 'group': 'All', 'kind': sg.KIND_NAMES[k],
                         'threshold': res['theta'], 'bound': res['bound'], 'calibration_accepted': res['accepted']})
    for a in G:
        r = np.full(len(pt['rows']), -1, dtype=np.int64)
        for g, labl in enumerate(G[a][1]):
            for k in (0, 1):
                hit, res = fit_release((yh_l == k) & (codes_l[a] == g), (pt['yhat'] == k) & (pt['codes'][a] == g),
                                       T['delta_2'] / (2 * len(G[a][1])), k)
                r[hit] = 0
                SGR_INFO.append({'task': task, 'method': 'SGR, per group', 'attribute': sg.ATTRIBUTE_NAMES[a], 'group': labl,
                                 'kind': sg.KIND_NAMES[k], 'threshold': res['theta'], 'bound': res['bound'],
                                 'calibration_accepted': res['accepted']})
        rel[('SGR, per group', a)] = r
    SGR_REL[task] = rel
    print(TASK_NAMES[task], f'SGR calibrated on {len(lab)} labelled records')
SGR_INFO = pd.DataFrame(SGR_INFO)

## Group results, policy tests per group and summaries

In [ ]:
ROWS = []
for task in TASKS:
    ft, st = FT[task], STATE[task]
    T = ft.targets
    alpha = {0: T['alpha_0'], 1: T['alpha_1']}
    alpha_sel = {0: T['alpha_sel']['negative_outcome'], 1: T['alpha_sel']['enrichment']}
    G = GROUPS[task]
    p2, pt = st['parts']['cal2'], st['parts']['test']
    for a, (_, labels) in G.items():
        runs = [('SCAX, marginal', REL[task]['cal2']['SCAX, marginal'], REL[task]['test']['SCAX, marginal'], len(labels)),
                ('SCAX, group-conditional', REL[task]['cal2'][('SCAX, group-conditional', a)],
                 REL[task]['test'][('SCAX, group-conditional', a)], len(labels)),
                ('SCAX, group-conditional, strict', REL[task]['cal2']['SCAX, group-conditional, strict'],
                 REL[task]['test']['SCAX, group-conditional, strict'], st['n_groups_all']),
                ('SGR', None, SGR_REL[task]['SGR'], None), ('SGR, per group', None, SGR_REL[task][('SGR, per group', a)], None)]
        for method, r2, rt, n_tests in runs:
            for k in (0, 1):
                met = sg.group_metrics(rt, pt['yhat'], pt['y'], pt['codes'][a], labels, k, st['size'] if r2 is not None else None)
                pol = sg.group_policy(r2, p2['yhat'], p2['y'], p2['codes'][a], len(labels), k, alpha_sel[k], T['delta_2'], n_tests) \
                    if r2 is not None else [{}] * len(labels)
                for i, row in enumerate(met):
                    extra = pol[i - 1] if i > 0 else {}
                    ROWS.append({'task': task, 'attribute': sg.ATTRIBUTE_NAMES[a], 'method': method, 'kind': sg.KIND_NAMES[k],
                                 'claim_target': alpha[k], 'alpha_sel': alpha_sel[k], **row, **extra})
RES = pd.DataFrame(ROWS)
RES.to_csv(f'{paths.RESULTS}/14_fairness.csv', index=False)

SUMMARY = []
for (task, attr, method, kind), d in RES.groupby(['task', 'attribute', 'method', 'kind'], sort=False):
    allrow, grp = d[d.group == 'All'].iloc[0], d[(d.group != 'All') & (d.patients > 0)]
    hi, lo = grp.loc[grp.certified_share.idxmax()], grp.loc[grp.certified_share.idxmin()]
    gap, glo, ghi = sg.newcombe_difference(int(hi.accepted), int(hi.patients), int(lo.accepted), int(lo.patients))
    acc = grp[grp.accepted > 0]
    worst = acc.loc[acc.error_rate.idxmax()] if len(acc) else None
    marg = RES[(RES.task == task) & (RES.attribute == attr) & (RES.method == 'SCAX, marginal') & (RES.kind == kind) & (RES.group == 'All')].iloc[0]
    SUMMARY.append({'task': task, 'attribute': attr, 'method': method, 'kind': kind, 'claim_target': allrow.claim_target,
                    'alpha_sel': allrow.alpha_sel, 'certified_share': allrow.certified_share,
                    'coverage_cost_vs_marginal': float(marg.certified_share - allrow.certified_share),
                    'error_rate': allrow.error_rate, 'highest_share_group': hi.group, 'lowest_share_group': lo.group,
                    'share_gap': gap, 'share_gap_lower_95': glo, 'share_gap_upper_95': ghi,
                    'worst_group': worst.group if worst is not None else None,
                    'worst_group_error_rate': float(worst.error_rate) if worst is not None else None,
                    'worst_group_error_lower_95': float(worst.error_lower_95) if worst is not None else None,
                    'worst_group_error_upper_95': float(worst.error_upper_95) if worst is not None else None,
                    'groups_above_claim_target': int((acc.error_rate > allrow.claim_target).sum()),
                    'groups_policy_certified': int(grp.policy_certified_in_group.fillna(False).astype(bool).sum())
                    if 'policy_certified_in_group' in grp and method.startswith('SCAX') else None,
                    'groups': int(len(grp))})
SUMMARY = pd.DataFrame(SUMMARY)
SUMMARY.to_csv(f'{paths.RESULTS}/14_fairness_summary.csv', index=False)
pd.set_option('display.width', 250)
show = ['task', 'attribute', 'method', 'kind', 'certified_share', 'coverage_cost_vs_marginal', 'error_rate', 'share_gap',
        'worst_group', 'worst_group_error_rate', 'worst_group_error_upper_95', 'groups_above_claim_target', 'groups_policy_certified', 'groups']
for kind in ('negative outcome', 'enrichment'):
    print(f'\n{kind} claims')
    print(SUMMARY[SUMMARY.kind == kind][show].round(3).to_string(index=False))

In [ ]:
show = ['group', 'patients', 'accepted', 'certified_share', 'error_rate', 'error_lower_95', 'error_upper_95', 'cal2_accepted',
        'U_sel_group', 'policy_certified_in_group']
for task in TASKS:
    for kind in ('negative outcome', 'enrichment'):
        d = RES[(RES.task == task) & (RES.kind == kind) & RES.method.isin(['SCAX, marginal', 'SCAX, group-conditional'])]
        print(f'\n{TASK_NAMES[task]}, {kind} claims (target {d.claim_target.iloc[0]})')
        print(d[['attribute', 'method'] + show].round(3).to_string(index=False))

## Figures: certified share and selective risk by group

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.lines import Line2D
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7.5, 'axes.spines.top': False, 'axes.spines.right': False})
STYLE = {'SCAX, marginal': ('SCAX, marginal', '#0072B2', 'o', True),
         'SCAX, group-conditional': ('SCAX, group-conditional', '#009E73', 's', True),
         'SCAX, group-conditional, strict': ('SCAX, group-conditional (strict)', '#D55E00', 'D', True),
         'SGR': ('SGR, no explanation', '#555555', 'o', False)}
OFF = dict(zip(STYLE, (-0.27, -0.09, 0.09, 0.27)))


def subgroup_figure(kind, fname):
    fig = plt.figure(figsize=(7.4, 6.4))
    gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 0.42, 1], hspace=0.42, wspace=0.1)   # spacer column before Cardio-70k
    axes = np.array([[fig.add_subplot(gs[i, c]) for c in (0, 1, 3)] for i in (0, 1)])
    for j, task in enumerate(TASKS):
        d = RES[(RES.task == task) & (RES.kind == kind) & (RES.group != 'All')]
        order = SIZES[SIZES.task == task][['attribute', 'group']].values.tolist()
        ypos = {tuple(g): len(order) - 1 - i for i, g in enumerate(order)}
        target, asel = d.claim_target.iloc[0], d.alpha_sel.iloc[0]
        for method, (lab, col, mk, filled) in STYLE.items():
            dm = d[d.method == method]
            yy = np.array([ypos[(a, g)] for a, g in zip(dm.attribute, dm.group)]) + OFF[method]
            kw = dict(marker=mk, ms=3.6, ls='none', color=col, mfc=col if filled else 'white', mec=col, mew=0.9)
            axes[0, j].plot(dm.certified_share, yy, **kw)
            ok = dm.accepted > 0
            if kind == 'enrichment':      # shown as the share with the outcome, the quantity the claim is about
                v, lo, hi = 1 - dm.error_rate[ok], 1 - dm.error_upper_95[ok], 1 - dm.error_lower_95[ok]
            else:
                v, lo, hi = dm.error_rate[ok], dm.error_lower_95[ok], dm.error_upper_95[ok]
            axes[1, j].errorbar(v, yy[ok.to_numpy()], xerr=[v - lo, hi - v], fmt=mk, ms=3.6, color=col,
                                mfc=col if filled else 'white', mec=col, mew=0.9, elinewidth=0.8, capsize=0)
        line = (1 - target) if kind == 'enrichment' else target
        axes[1, j].axvline(line, color='0.2', ls='--', lw=0.9)
        if not np.isclose(asel, target):
            axes[1, j].axvline((1 - asel) if kind == 'enrichment' else asel, color='0.2', ls=':', lw=0.9)
        for i in (0, 1):
            ax = axes[i, j]
            ax.set_yticks(list(ypos.values()))
            same = j > 0 and order == SIZES[SIZES.task == TASKS[j - 1]][['attribute', 'group']].values.tolist()
            ax.set_yticklabels([] if same else [g.replace('$', r'\$') for _, g in ypos.keys()])   # $ is not mathtext
            ax.set_ylim(-0.6, len(order) - 0.4)
            ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
            ax.grid(axis='x', color='0.92', lw=0.6); ax.set_axisbelow(True)
            for b in range(1, len(order)):
                if order[b][0] != order[b - 1][0]:
                    ax.axhline(len(order) - b - 0.5, color='0.8', lw=0.6)
        axes[0, j].set_title(TASK_NAMES[task], fontsize=8.5)
        axes[0, j].set_xlim(left=-0.03 * axes[0, j].get_xlim()[1])      # markers at 0% stay visible
    what = 'negative-outcome' if kind == 'negative outcome' else 'enrichment'
    axes[0, 1].set_xlabel(f'Share of the group given a certified {what} claim')
    axes[1, 1].set_xlabel('Share with the outcome among patients given a claim')
    handles = [Line2D([], [], marker=mk, ls='none', color=col, mfc=col if f else 'white', mec=col, ms=4.5, label=lab)
               for lab, col, mk, f in STYLE.values()]
    handles += [Line2D([], [], color='0.2', ls='--', lw=0.9, label='Claim target'),
                Line2D([], [], color='0.2', ls=':', lw=0.9, label='Policy target, where different')]
    fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.subplots_adjust(left=0.17, right=0.98, top=0.95, bottom=0.14)
    for ext in ('png', 'pdf'):
        fig.savefig(paths.FIGURES / f'{fname}.{ext}', dpi=600, bbox_inches='tight')
    plt.show()


subgroup_figure('negative outcome', 'fig_subgroups_negative_outcome')
subgroup_figure('enrichment', 'fig_subgroups_enrichment')

In [ ]:
SGR_INFO.to_csv(f'{paths.RESULTS}/14_sgr_thresholds.csv', index=False)
write_json(f'{paths.RESULTS}/14_subgroups.json', {
    'frozen_config_sha256': FROZEN_SHA, 'box_table_sha256': BOX_SHA, 'limit': LIMIT,
    'income_cuts': {t: c for t, c in CUTS.items() if c is not None},
    'groups': {t: {sg.ATTRIBUTE_NAMES[a]: l for a, (_, l) in GROUPS[t].items()} for t in TASKS},
    'second_check': VERIFY,
    'summary': SUMMARY.to_dict('records')})
print('written: results/14_group_sizes.csv, 14_fairness.csv, 14_fairness_summary.csv, 14_sgr_thresholds.csv, 14_subgroups.json; '
      'figures/fig_subgroups_negative_outcome, fig_subgroups_enrichment')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "14: subgroups (sex, age band, income tertile) for CDC Diabetes, BRFSS Heart and Cardio-70k: marginal and group-conditional SCAX (standard and strict), SGR, group policy tests, worst-group risk, figures"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)